#### Extraction

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
 
from Extraction import extract
from Transformation_Pretraitement import preprocessing_polars
from inceptionTimeModified import (
    evaluate_on_test,
    load_model_from_checkpoint,
    predict_proba,
    train_inception_time,
)

In [ ]:
pl.Config.set_tbl_cols(-1)

 ##### Dataframe statique

In [ ]:

path = "../Datasets/clean_full_static_ano.parquet"
df_static = pl.read_parquet(path)
df_static = df_static.with_columns(pl.col("encounterId").cast(pl.Int32))

##### Dataframe dynamique

In [ ]:

path = "../Datasets/df_with_calculated_features.parquet"
df_test = extract.extract_data_survie(path)

#### Transformation_Prétraitement

Il faudra changer hour_offset pour pouvoir prendre une date fixe et non juste un temps en arrière

ajout de la colonne age qui est dans le thesaurus

In [ ]:
df_test = df_test.join(
    df_static[["encounterId", "age"]],
    on="encounterId",
    how="left"
)

On utilise ici le preprocessing de Gabrielle mais avec l'optimisation polars réalisée par mes soins, puisque l'ancien code mettait beaucoup trop de temps à tourner.

In [ ]:

df_clean = preprocessing_polars.prepare_data(df_test,0)

In [ ]:
df_with_idx = df_clean.with_row_index("idx")
 
idx = (

    df_with_idx

    .filter(pl.col("fio2_corr").is_null())

    .select("idx")

)
 
print(len(idx))

df_clean.filter(pl.col("fio2_corr").is_null())

Il faut rajouter isDeceased sinon on n'a pas de Y

In [ ]:
df_clean = df_clean.join(
    df_static[["encounterId", "isDeceased"]],
    on="encounterId",
    how="left"
)

#### Préparation pour InceptionTime

##### Fonctions utilisées

**scaling :** Fonction qui permet de scaler en utilisant StandardScaler, uniquement les entiers et pas les booléens

In [ ]:
def scaling(df_train, df_test):
    # StandardScaler ne supporte pas directement Polars mais uniquement pandas donc transfert obligatoire...
    df_train_pd = df_train.to_pandas()
    df_test_pd = df_test.to_pandas()
    
    # colonnes numériques sauf bool
    num_cols = df_train_pd.select_dtypes(include=["number"]).columns
    bool_cols = df_train_pd.select_dtypes(include=["bool"]).columns
    
    num_cols = [c for c in num_cols if c not in bool_cols and c not in [patient_col, target_col]]
    
    scaler = StandardScaler()
    
    df_train_pd[num_cols] = scaler.fit_transform(df_train_pd[num_cols])
    df_test_pd[num_cols] = scaler.transform(df_test_pd[num_cols])
    return df_train_pd, df_test_pd

**build_sequences :** Fonction qui permet d'extraire des dataframes train et test, les features appropriées et les split entre X et y

In [ ]:
def build_sequences(df, patient_col, target_col, expected_length):
    X_list = []
    y_list = []
 
    for subdf in df.partition_by(patient_col, maintain_order=True):
        if subdf.height != expected_length:
            raise ValueError(f"Le groupe {subdf[patient_col][0]} n'a pas {expected_length} lignes.")
 
        X_list.append(subdf.select(pl.exclude(target_col, patient_col)).to_numpy())
        y_list.append(subdf[target_col][0])  # un seul label par séquence
 
    X_3d = np.stack(X_list).astype(np.float32)
    y_1d = np.array(y_list).astype(np.int64)
 
    return X_3d, y_1d

##### Split train/test + scaling + reshape

In [ ]:
patient_col="encounterId"
time_col="heure_calibree"
target_col="isDeceased"
expected_length=24

# On garde les encounters de longueur exacte 
valid_ids = (df_clean.group_by(patient_col).len()
        .filter(pl.col("len") == expected_length)
        .select(patient_col))

df_clean = df_clean.join(valid_ids, on=patient_col, how="inner")

if df_clean.is_empty():
    raise ValueError("Aucun patient n'a exactement la longueur attendue.")

# Tri obligatoire (même si en théorie il est déjà fait)
df_clean = df_clean.sort(patient_col, time_col)

# On prépare le jeu d'entraînement
X = df_clean.select(pl.exclude(target_col, patient_col)).to_numpy()
y = df_clean[target_col].to_numpy()
groups = df_clean[patient_col].to_numpy() # grouper en fonction d'un individu

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# On prend un split (comme train/test mais adapté aux individus)
train_idx, test_idx = next(sgkf.split(X=X, y=y, groups=groups))

# Normalement pas besoin de sort mais soyons prudents...
train_df = df_clean[train_idx].sort([patient_col, time_col])
test_df = df_clean[test_idx].sort([patient_col, time_col])
# Ok maintenant, on applique le scaler sur le dataframe

train_pd, test_pd = scaling(train_df, test_df)

# On retransforme en df polars
train_df, test_df = pl.from_pandas(train_pd), pl.from_pandas(test_pd)

# On construit la séquence attendue (N, T, F) à partir des deux dataframes train/test
X_train_3d, y_train_seq = build_sequences(
    train_df, patient_col, target_col, expected_length
)

X_test_3d, y_test_seq = build_sequences(
    test_df, patient_col, target_col, expected_length
)

# TODO vérifier si y'a bien 24 

In [ ]:
# TODO : print(shape[0]%24 == 0)

In [ ]:
train_df.group_by("encounterId").len().describe()

#### Recherche des meilleurs hyperparamètres avec Optuna

In [ ]:
import optuna
 
study = optuna.create_study(
    study_name="test_optuna",
    direction="minimize",
    storage="sqlite:///optuna_inception2.db",
    load_if_exists=True,
)
 
study.optimize(lambda trial: trial.suggest_float("x", 0, 1), n_trials=1)

In [ ]:
import sqlite3
print(sqlite3.sqlite_version)

In [ ]:
import math
import optuna
import numpy as np

def extract_best_val_loss(history):
    if history is None:
        raise ValueError("history est None.")
    if "val_loss" not in history:
        raise ValueError("La clé 'val_loss' est absente de history.")
 
    values = [v for v in history["val_loss"] if v is not None and np.isfinite(v)]
    if not values:
        raise ValueError("Aucune val_loss valide trouvée.")
 
    return float(min(values))
 
 
# --------------------------------------------------
# Objective - recherche large
# --------------------------------------------------
def make_objective_stage1(
    X_train_3d,
    y_train_seq,
    metric_name="val_loss",
    fixed_params=None,
):
    fixed_params = fixed_params or {}
 
    def objective(trial):
        params = {
            "val_ratio": fixed_params.get("val_ratio", 0.2),
            "epochs": fixed_params.get("epochs", 100),
            "patience": fixed_params.get("patience", 10),
            "min_delta": fixed_params.get("min_delta", 0.0),
            "calibrate": fixed_params.get("calibrate", False),
            "save_best_path": None,
            "device": fixed_params.get("device", "cuda"),
            "progress": False,
 
            # Hyperparams à explorer largement
            "num_blocks": trial.suggest_int("num_blocks", 3, 8),
            "out_channels": trial.suggest_categorical("out_channels", [16, 32, 64, 128]),
            "bottleneck_channels": trial.suggest_categorical("bottleneck_channels", [8, 16, 32, 64]),
            "kernel_sizes": trial.suggest_categorical("kernel_sizes", [15, 21, 31, 41, 51, 61]),
            "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64, 128]),
            "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
            "clip_grad": trial.suggest_categorical("clip_grad", [0.5, 1.0, 2.0]),
            "use_scheduler": trial.suggest_categorical("use_scheduler", [True, False]),
        }
 
        try:
            model, T, history, splits = train_inception_time(
                X_train_3d,
                y_train_seq,
                **params,
            )
 
            score = extract_best_val_loss(history)
 
            if not np.isfinite(score):
                raise FloatingPointError("Score non fini.")
 
            return score
 
        except FloatingPointError:
            raise
        except Exception as e:
            # si un essai plante, on le marque comme mauvais essai
            raise optuna.TrialPruned(f"Trial échoué: {e}")
 
    return objective
 
 
# --------------------------------------------------
# Lancement étude 1
# --------------------------------------------------
def run_stage1_search(
    X_train_3d,
    y_train_seq,
    n_trials=40,
    study_name="inception_stage1",
    storage=None,
    metric_name="val_loss",
    fixed_params=None,
):
    sampler = optuna.samplers.TPESampler(seed=42)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=5)
 
    study = optuna.create_study(
        study_name=study_name,
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
        storage=storage,
        load_if_exists=True,
    )
 
    objective = make_objective_stage1(
        X_train_3d=X_train_3d,
        y_train_seq=y_train_seq,
        metric_name=metric_name,
        fixed_params=fixed_params,
    )
 
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True)
 
    print("=== STAGE 1 ===")
    print("Best value :", study.best_value)
    print("Best params:", study.best_params)
 
    return study

In [ ]:
fixed_params = {
    "val_ratio": 0.2,
    "epochs": 100,
    "patience": 10,
    "min_delta": 0.0,
    "calibrate": False,
    "device": "cuda",
}
 
study_stage1 = run_stage1_search(
    X_train_3d,
    y_train_seq,
    n_trials=40,
    study_name="inception_stage1",
    storage="sqlite:///optuna_inception2.db",
    metric_name="val_loss",   # ou "val_f1", selon ton history
    fixed_params=fixed_params,
)

In [ ]:
import optuna
import numpy as np
 
 
def _neighbors_from_choices(best_value, choices):
    """
    Pour un hyperparam catégoriel ordinal, on prend le voisin du dessous,
    lui-même, et le voisin du dessus.
    """
    choices = list(choices)
    idx = choices.index(best_value)
    low_idx = max(0, idx - 1)
    high_idx = min(len(choices) - 1, idx + 1)
    return choices[low_idx:high_idx + 1]
 
 
def make_objective_stage2(
    X_train_3d,
    y_train_seq,
    best_stage1_params,
    metric_name="val_loss",
    fixed_params=None,
):
    fixed_params = fixed_params or {}
 
    # espaces globaux de référence
    out_choices = [16, 32, 64, 128]
    bottleneck_choices = [8, 16, 32, 64]
    kernel_choices = [15, 21, 31, 41, 51, 61]
    batch_choices = [16, 32, 64, 128]
    clip_choices = [0.5, 1.0, 2.0]
 
    # bornes fines autour du meilleur stage1
    best_lr = best_stage1_params["lr"]
    lr_low = max(1e-5, best_lr / 3.0)
    lr_high = min(1e-2, best_lr * 3.0)
 
    best_wd = best_stage1_params["weight_decay"]
    wd_low = max(1e-7, best_wd / 10.0)
    wd_high = min(1e-1, best_wd * 10.0)
 
    num_blocks_best = best_stage1_params["num_blocks"]
    num_blocks_low = max(2, num_blocks_best - 1)
    num_blocks_high = min(10, num_blocks_best + 1)
 
    out_candidates = _neighbors_from_choices(best_stage1_params["out_channels"], out_choices)
    bottleneck_candidates = _neighbors_from_choices(best_stage1_params["bottleneck_channels"], bottleneck_choices)
    kernel_candidates = _neighbors_from_choices(best_stage1_params["kernel_sizes"], kernel_choices)
    batch_candidates = _neighbors_from_choices(best_stage1_params["batch_size"], batch_choices)
    clip_candidates = _neighbors_from_choices(best_stage1_params["clip_grad"], clip_choices)
 
    scheduler_best = best_stage1_params["use_scheduler"]
 
    def objective(trial):
        params = {
            "val_ratio": fixed_params.get("val_ratio", 0.2),
            "epochs": fixed_params.get("epochs", 100),
            "patience": fixed_params.get("patience", 10),
            "min_delta": fixed_params.get("min_delta", 0.0),
            "calibrate": fixed_params.get("calibrate", False),
            "save_best_path": None,
            "device": fixed_params.get("device", "cuda"),
            "progress": False,
 
            # recherche fine
            "num_blocks": trial.suggest_int("num_blocks", num_blocks_low, num_blocks_high),
            "out_channels": trial.suggest_categorical("out_channels", out_candidates),
            "bottleneck_channels": trial.suggest_categorical("bottleneck_channels", bottleneck_candidates),
            "kernel_sizes": trial.suggest_categorical("kernel_sizes", kernel_candidates),
            "batch_size": trial.suggest_categorical("batch_size", batch_candidates),
            "lr": trial.suggest_float("lr", lr_low, lr_high, log=True),
            "weight_decay": trial.suggest_float("weight_decay", wd_low, wd_high, log=True),
            "clip_grad": trial.suggest_categorical("clip_grad", clip_candidates),
            "use_scheduler": trial.suggest_categorical("use_scheduler", [scheduler_best]),
        }
 
        try:
            model, T, history, splits = train_inception_time(
                X_train_3d,
                y_train_seq,
                **params,
            )
 
            score = extract_best_val_loss(history)
 
            if not np.isfinite(score):
                raise FloatingPointError("Score non fini.")
 
            return score
 
        except FloatingPointError:
            raise
        except Exception as e:
            raise optuna.TrialPruned(f"Trial échoué: {e}")
 
    return objective
 
 
def run_stage2_search(
    X_train_3d,
    y_train_seq,
    study_stage1,
    n_trials=25,
    study_name="inception_stage2",
    storage=None,
    metric_name="val_loss",
    fixed_params=None,
):
    best_stage1_params = study_stage1.best_params
 
    sampler = optuna.samplers.TPESampler(seed=43)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
 
    study = optuna.create_study(
        study_name=study_name,
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
        storage=storage,
        load_if_exists=True,
    )
 
    objective = make_objective_stage2(
        X_train_3d=X_train_3d,
        y_train_seq=y_train_seq,
        best_stage1_params=best_stage1_params,
        metric_name=metric_name,
        fixed_params=fixed_params,
    )
 
    study.optimize(objective, n_trials=n_trials, gc_after_trial=True)
 
    print("=== STAGE 2 ===")
    print("Best value :", study.best_value)
    print("Best params:", study.best_params)
 
    return study

#### Training sur InceptionTime

In [ ]:
model, T, history, splits = train_inception_time(
    X_train_3d, y_train_seq,
    epochs=100,
    patience=10,
    save_best_path="models/inception_test2.pt"
)

In [ ]:
print(history.keys())

#### Evaluation des résultats obtenus

In [ ]:
auc, brier, T = evaluate_on_test(
    X_test_3d, y_test_seq,
    "models/inception_test2.pt"
)



model, _, T = load_model_from_checkpoint("models/inception_test2.pt")

In [ ]:
probas = predict_proba(model, X_test_3d, T=T)
print(probas)

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test_seq, probas)
auc = roc_auc_score(y_test_seq, probas)
 
plt.figure(figsize=(6, 6))
plt.plot(fpr, tpr, label=f"ROC (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Hasard")
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs")
plt.title("Courbe ROC")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
plt.figure()
 
sns.kdeplot(probas[y_test_seq == 0], label="Survivants", fill=True)
sns.kdeplot(probas[y_test_seq == 1], label="Décès", fill=True)
 
plt.xlabel("Probabilité prédite")
plt.ylabel("Densité")
plt.title("Distribution des scores (KDE)")
plt.legend()
plt.grid()
plt.show()

In [ ]:
y_test = y_test_seq

plt.figure()
 
# Survivants
data_0 = probas[y_test == 0]
sns.kdeplot(data_0, color="lightblue")
x0, y0 = plt.gca().lines[-1].get_data()
y0 = y0 * len(data_0)  # conversion densité → counts
plt.plot(x0, y0, color="blue", label="Survivants")
plt.fill_between(x0, y0, alpha=0.3, color="lightblue")
 
# Décès
data_1 = probas[y_test == 1]
sns.kdeplot(data_1, color="orange")
x1, y1 = plt.gca().lines[-1].get_data()
y1 = y1 * len(data_1)
plt.plot(x1, y1, color="orange", label="Décès")
plt.fill_between(x1, y1, alpha=0.3, color="orange")
 
plt.xlabel("Probabilité prédite")
plt.ylabel("Nombre de patients")
plt.title("Distribution des scores")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.style.use("seaborn-v0_8")
plt.show()

In [ ]:

prob_true, prob_pred = calibration_curve(y_test, probas, n_bins=10)
 
plt.figure()
plt.plot(prob_pred, prob_true, marker="o", label="Modèle")
plt.plot([0, 1], [0, 1], "--", label="Calibration idéale")
 
plt.xlabel("Probabilité prédite")
plt.ylabel("Fréquence observée")
plt.title("Calibration curve")
plt.legend()
plt.grid()
plt.show()

In [ ]:

thresholds = np.linspace(0.1, 0.9, 50)
f1s = []
best_f1 = 0
for t in thresholds:
    y_pred = (probas >= t).astype(int)
    f1s.append(f1_score(y_test, y_pred))
    f1 = f1_score(y_test, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_t = t
 
import matplotlib.pyplot as plt
 
plt.plot(thresholds, f1s)
plt.xlabel("Threshold")
plt.ylabel("F1 score")
plt.title("F1 vs Threshold")
plt.grid()
plt.show()

print(f"Le meilleur f1 score de{best_f1 : .2f} est atteint lorsque le threshold est égal à{best_t : .2f}")

In [ ]:
threshold = 0.46
y_pred = (probas >= threshold).astype(int)
 
cm = confusion_matrix(y_test, y_pred)
 
plt.figure()
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Prédit")
plt.ylabel("Réel")
plt.title(f"Confusion matrix (threshold={threshold})")
plt.show()